# Model 2 — 약관 조문을 조건식 JSON 으로

은행 상품설명서의 조문을 **기계가 실행할 수 있는 조건식 JSON** 으로 옮겨 3번 모델에 넘긴다.

```
상품설명서 ─▶ 조문 분할 ─▶ 조건식 생성 ─▶ 검증 · 재시도 ─▶ 조건식 JSON ─▶ 3번 모델 (문장화)
              (코드)        (Qwen3-8B)      (코드)           (+ 고객 입력으로 걸러 낸 조건부 묶음)
```

**이 단계의 범위는 조건식까지다** (2026-10-08 결정). 사람이 읽는 문장은 3번 모델이 만든다.
LLM 은 «조문 → 조건식» 한 곳에만 쓰고, 나머지는 전부 결정론적 코드다.

| 절 | 내용 | GPU |
|---|---|---|
| 0 | 환경 확인 | |
| 1 | 조문 분할 | |
| 2 | 카탈로그 — 쓸 수 있는 이름 | |
| 3 | 프롬프트와 재시도 | (§3.3 데모만) |
| 4 | 출력 형식 — 조건식 DSL | |
| 5 | 평가 — 형식 관문과 뜻 | |
| 6 | 3번 모델로 넘기는 것 | |
| 7 | 지금까지 배운 것 (이력 요약) | |
| 8 | 한계와 팀이 정할 것 | |

모든 셀은 저장소 `model/M2-2` 의 `.py` 모듈과 저장된 결과 파일을 그대로 읽는다.
GPU 가 드는 셀은 §3.3 하나이고 기본값은 꺼져 있다.

**기준 데이터** — KB 상품 10종 상품설명서 105조문(이하 kb10), 정답 105조문(사람 검토 완료 10-06),
마지막 코드로 돌린 결과 두 판(`results_kb10v127a/b.json`).

---
## 0. 환경 확인

커널은 `/workspace/.venv` 여야 한다. 시스템 python 의 transformers 는 Qwen3 tokenizer 를 못 읽는다.

In [1]:
import os, sys, json, collections, warnings
from pathlib import Path
warnings.filterwarnings("ignore", message="IProgress not found")

REPO_ROOT = os.environ.get("M2_REPO_ROOT", "/workspace/personal_workplace_CHO/model/M2-2")
assert Path(REPO_ROOT, "catalog.py").is_file(), f"저장소 없음: {REPO_ROOT}"
os.chdir(REPO_ROOT)
for p in (REPO_ROOT, str(Path(REPO_ROOT, "showcase"))):
    if p not in sys.path:
        sys.path.insert(0, p)

import transformers
print("python      ", sys.executable)
print("transformers", transformers.__version__)
assert sys.executable.startswith("/workspace/.venv"), "커널을 /workspace/.venv 로 바꿀 것"

python       /workspace/.venv/bin/python
transformers 5.18.0.dev0


---
## 1. 조문 분할 (LLM 없음)

상품설명서를 조문 단위로 자른다. **모델을 쓰지 않는다** — 규칙으로 자르면 매번 같은 결과가 나오고,
전처리가 흔들리면 그 아래 측정이 전부 흔들린다.

| | |
|---|---|
| 섹션 · 표 인식 | 헤딩과 표를 블록으로 나눈다 |
| 참조표 인라인 | «아래 표 참조» 자리에 그 표를 끌어와 붙인다 — 모델이 표를 못 보면 조건식을 못 만든다 |
| 중복 제거 | 붙인 표는 단독 조문에서 뺀다 — 남기면 같은 내용이 두 번 채점된다(초기 코퍼스의 19%가 그랬다) |
| 정형문구 제외 | 예금자보호 안내 같은 boilerplate |

kb10 조문은 `showcase/build_clauses.py` 가 DB 의 상품설명서 항목에서 떴다(DB 는 지금 서버에 없어
다시 뜰 수 없다). 결과는 `showcase/clauses_key.json` 에 있다.

In [2]:
CLAUSES = json.loads(Path("showcase/clauses_key.json").read_text("utf-8"))
PRODUCTS = {p["prefix"]: p for p in json.loads(Path("showcase/products.json").read_text("utf-8"))}
CL = {c["id"]: c for c in CLAUSES}

per = collections.Counter(c["id"].split("-")[0] for c in CLAUSES)
print(f"kb10 {len(CLAUSES)}조문 · 상품 {len(per)}종\n")
for k, n in per.most_common():
    p = PRODUCTS[k]
    print(f"  {k:<6}{p['kind']:<6}{n:>3}조문  {p['product']}")
print("\n분류:", dict(collections.Counter(c["category"] for c in CLAUSES).most_common()))

kb10 105조문 · 상품 10종

  KBSSG 입출금    17조문  쓱머니KB통장
  KBMY  적금     16조문  KB내맘대로적금
  KBSME 적금     15조문  KB 중소기업재직자 우대저축
  KBGL  입출금    11조문  KB골든라이프연금우대통장
  KBNR  입출금    10조문  KB나라사랑우대통장
  KBMER 적금     10조문  KB가맹점우대적금
  KBHAP 적금      9조문  KB국민행복적금
  KBSTR 정기예금    9조문  KB Star 정기예금
  KBSUP 정기예금    5조문  국민수퍼정기예금(개인)
  KBGEN 정기예금    3조문  일반정기예금

분류: {'rate': 57, 'eligibility': 24, 'benefit': 24}


이 노트북에서 계속 쓸 예시 조문이다 — 급여이체 실적에 따른 우대이율. 거래 내역이 있어야 판정되는
조건이라 §6 의 «조건부»까지 이어진다.

In [3]:
DEMO = "KBNR-S2-R05"
c = CL[DEMO]
print(f"[{c['id']}] {c['label']}  ({PRODUCTS['KBNR']['product']}, {c['category']}, {len(c['text'])}자)")
print("-" * 78)
print(c["text"])

[KBNR-S2-R05] 우대이율 (2026.03.09 기준 세금공제 전)  (KB나라사랑우대통장, rate, 402자)
------------------------------------------------------------------------------
[우대이율 (2026.03.09 기준 세금공제 전)]
▪ 우대이율 : 연 1.9%p
▪ 적용대상 : 결산일의 전월 말일을 기준으로 직전 3개월 동안 2개월 이상 이 통장으로 아래 한 항목 이상에 해당하는 급여이체 실적이 있는 경우 • 이 통장으로 KB국민은행과 급여이체 계약에 의한 국군재정관리단을 통한 급여이체 실적이 있는 경우 • 이 통장으로 육군「통합자금관리시스템」에 의한 급여이체 실적이 있는 경우 • 군인공제회에서 KB국민은행으로 제공하는 급여자료 중 이 통장으로 급여이체 실적이 있는 경우
※ 각 호 이외의 급여이체는 우대이율이 적용되지 않습니다.
▪ 적용한도 : 이 통장의 결산기 평균잔액 중 100만원 이하의 금액까지는 기본이율과 우대이율을 합산하여 적용하며, 100만원을 초과하는 금액은 기본이율만 적용


---
## 2. 카탈로그 — 쓸 수 있는 이름

**모델은 카탈로그에 있는 이름만 쓸 수 있다.** 벗어나면 관문 ③ 에서 떨어진다. 자유 서술을 허용하면
같은 뜻이 열 가지 표기로 갈리고, 실행기를 만들 수 없다.

카탈로그를 넓힐지가 반복되는 판단이다. 기준은 하나다 — **사건이 아니라 상태 비교인 것을 이벤트로
받아 주지 않는다.** 모델이 지어낸 이름을 받아 주면 모델링 오류가 카탈로그에 굳는다.

프롬프트에는 카탈로그 전체가 아니라 **조문별 후보만** 넣는다. 관계없는 이름은 오답 후보가 된다.

In [4]:
import catalog

print(f"변수 {len(catalog.VARIABLES)} · 이벤트 {len(catalog.EVENTS)} · 연산자 {len(catalog.OPERATORS)}"
      f" · effect {len(catalog.EFFECTS)} · 면제 대상 {len(catalog.FEES)}"
      f" · 서비스 {len(catalog.SERVICES)} · 제한 {len(catalog.RESTRICTIONS)}\n")
print("effect :", ", ".join(catalog.EFFECTS))
print("이벤트 :", ", ".join(catalog.EVENTS))

v, e = catalog.pick_candidates(CL[DEMO]["text"])
print(f"\n{DEMO} 에 넣는 후보 — 변수 {len(v)}개, 이벤트 {len(e)}개")
print("  이벤트:", ", ".join(e))

변수 22 · 이벤트 12 · 연산자 15 · effect 15 · 면제 대상 6 · 서비스 3 · 제한 9

effect : RATE_ADD, RATE_SUBTRACT, RATE_OVERRIDE, PENALTY_RATE, DISQUALIFY, LIMIT_AMOUNT, REQUIRE, FEE_WAIVE, SERVICE_PROVIDE, FEE_WAIVE_NO_CARRYOVER, DORMANT_CONTRIBUTION, RESTRICT, TAX_EXEMPT_ELIGIBLE, PRODUCT_SWITCH_TO, STATE_CHANGE
이벤트 : SALARY_TRANSFER, AUTO_TRANSFER, CARD_USAGE, DEPOSIT, WITHDRAWAL, PENSION_RECEIPT, FX_TRANSACTION, PRODUCT_SWITCH, TERMINATION, APP_LOGIN, TRANSACTION_CORRECTION, NAME_CHANGE

KBNR-S2-R05 에 넣는 후보 — 변수 5개, 이벤트 1개
  이벤트: SALARY_TRANSFER


---
## 3. 프롬프트와 재시도

### 3.1 첫 프롬프트

`SYSTEM_PROMPT` (규칙 · 표기법) + `build_user_prompt()` (few-shot · 변환할 조문 · 그 조문의 후보 이름).
추론 모드는 끈다 — 매핑 태스크다.

In [5]:
from prompt import SYSTEM_PROMPT, build_user_prompt
from transformers import AutoTokenizer

user = build_user_prompt(CL[DEMO]["text"], DEMO)
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B")
text = tok.apply_chat_template([{"role": "system", "content": SYSTEM_PROMPT},
                                {"role": "user", "content": user}],
                               tokenize=False, add_generation_prompt=True, enable_thinking=False)
print(f"SYSTEM_PROMPT {len(SYSTEM_PROMPT):>6,}자")
print(f"user prompt   {len(user):>6,}자")
print(f"합계          {len(tok(text)['input_ids']):>6,} 토큰 · 생성 한도 3,072 토큰")

SYSTEM_PROMPT    921자
user prompt    5,205자
합계           2,736 토큰 · 생성 한도 3,072 토큰


### 3.2 검증 → 재시도

생성물을 관문 다섯(§5)과 뜻 검사(`validate.semantic_errors`)에 태우고, 걸리면 **오류를 모두 모아**
재시도 프롬프트에 넣는다(최대 3회). 오류를 하나만 알려 주면 나머지를 못 고친다.

**재시도 중 가장 나은 시도를 낸다 — 마지막 것이 아니다.** 순서는 «통과한 관문 수 → 뜻 오류가 적은 것
→ 오류가 적은 것». 고치라고 했는데 더 나빠진 답을 받아들이지 않는다.

그 뒤 후처리 둘이 붙는다 — 원문에 없는 숫자를 지어낸 rule 걷어 내기(`postfilter.py`),
빈 배열이면 조문 머리말로 이유 붙이기(`reasons.py`).

### 3.3 모델 실행 ⚠️ GPU 필요

기본은 꺼져 있다. 켜면 예시 조문 하나를 생성한다(모델 로딩 약 80초 + 생성). GPU 여유가 24GB 는 있어야 한다.
전체 라운드는 터미널에서 돈다.

```bash
CORPUS=showcase/clauses_key.json ./showcase/run_showcase.sh <이름>   # kb10 105조문, 약 2.5시간
```

In [6]:
RUN_MODEL = False

if RUN_MODEL:
    from run_local_v2 import load_model, run_clause
    tok_m, model = load_model("Qwen/Qwen3-8B")
    rec = run_clause(tok_m, model, CL[DEMO], runs=1)[0]
    print(json.dumps(rec["output"], ensure_ascii=False, indent=1))
    print({k: rec["final"][k] for k in ("json_ok", "schema_ok", "catalog_ok", "quote_ok", "exec_ok")})
else:
    print("건너뜀 — 아래 절은 저장된 결과로 돈다")

건너뜀 — 아래 절은 저장된 결과로 돈다


---
## 4. 출력 형식 — 조건식 DSL

조문 하나 → `{"rules": [...]}`. rule 하나의 층:

| 키 | 뜻 |
|---|---|
| `source.quote` | 근거가 된 **원문 그대로** — 지어내면 관문 ④ 에서 걸린다 |
| `let` | 기간 정의. 묶을 때는 `window`, 참조할 때만 `{"ref": "$window"}` |
| `guard` | 이 조항을 **볼지 말지** (예: 중도해지면 우대 조항을 아예 안 본다) |
| `condition` | 충족 여부 — 연산자 이름이 키다: `{">=": [a, b]}` |
| `effect` | 충족했을 때 일어나는 일 (`RATE_ADD` · `FEE_WAIVE` · `DISQUALIFY` …) |
| `status` | `OK` 실행 가능 · `UNSUPPORTED` 카탈로그로 표현 불가(이유를 `label` 에) |

옮길 것이 없는 조문(상품 개요 · 안내문)은 **빈 배열이 정답**이다. 이때 `no_rule_reason` 을 붙인다.

예시 조문의 정답이다. 두 번째 rule 은 «평균잔액 100만원 이하분에만 우대»라 표현할 effect 가 없어
`UNSUPPORTED` 로 남겼다 — 억지로 옮기지 않는다.

In [7]:
GOLD = {p.stem: json.loads(p.read_text("utf-8")) for p in sorted(Path("showcase/gold_draft").glob("*.json"))}
print(json.dumps(GOLD[DEMO]["rules"], ensure_ascii=False, indent=1))

[
 {
  "rule_id": "KBNR-S2-R05-01",
  "status": "OK",
  "label": "직전 3개월 중 2개월 이상 급여이체 실적 → 우대이율 연 1.9%p",
  "source": {
   "quote": "▪ 우대이율 : 연 1.9%p ▪ 적용대상 : 결산일의 전월 말일을 기준으로 직전 3개월 동안 2개월 이상 이 통장으로 아래 한 항목 이상에 해당하는 급여이체 실적이 있는 경우"
  },
  "let": {
   "window": {
    "month_range": {
     "from": {
      "anchor": "contract.reference_date",
      "offset_months": -3,
      "day": "first"
     },
     "to": {
      "anchor": "contract.reference_date",
      "offset_months": -1,
      "day": "last"
     }
    }
   }
  },
  "condition": {
   ">=": [
    {
     "count_distinct_months": {
      "event": "SALARY_TRANSFER",
      "within": {
       "ref": "$window"
      }
     }
    },
    2
   ]
  },
  "effect": {
   "type": "RATE_ADD",
   "value": 1.9,
   "unit": "%p",
   "applies_to": "BASE_RATE"
  }
 },
 {
  "rule_id": "KBNR-S2-R05-02",
  "status": "UNSUPPORTED",
  "label": "평균잔액 100만원 이하분에만 우대이율 — 적용 기준 금액(잔액 구간) effect 없음",
  "source": {
   "quote": "▪ 적용한도 : 이 통장의 결산기 평균잔액 중 100만원 이하

---
## 5. 평가 — 형식 관문과 뜻

### 5.1 형식 관문 ①~⑤

정답 없이 잴 수 있는 것들이다. 빈 배열은 ③④⑤ 를 공짜로 통과하지 못하게 따로 뗀다(⑦).

| | 지표 | 묻는 것 |
|---|---|---|
| ① | `json_ok` | JSON 으로 읽히는가 |
| ② | `schema_ok` | 층 구조를 지키는가 |
| ③ | `catalog_ok` | 카탈로그 밖의 이름을 지어내는가 |
| ④ | `quote_ok` | 근거 인용이 원문과 일치하는가 |
| ⑤ | `exec_ok` | 실행기에 태웠을 때 오류 없이 끝나는가 |
| ⑦ | `empty` | 빈 배열을 냈는가 (별도 축) |

In [8]:
RUNS = {k: json.loads(Path(f"showcase/results_kb10v127{k}.json").read_text("utf-8")) for k in "ab"}
GATES = ("json_ok", "schema_ok", "catalog_ok", "quote_ok", "exec_ok")

print(f"{'':14}" + "".join(f"{'v127' + k:>10}" for k in RUNS))
print(f"{'⑦ 빈 배열':14}" + "".join(f"{sum(x['final']['empty'] for x in R) / len(R):>10.1%}" for R in RUNS.values()))
for g in GATES:
    print(f"{g:14}" + "".join(
        f"{sum(x['final'][g] for x in R if g in ('json_ok', 'schema_ok') or not x['final']['empty']) / (len(R) if g in ('json_ok', 'schema_ok') else sum(not x['final']['empty'] for x in R)):>10.1%}"
        for R in RUNS.values()))
print("\n③④⑤ 는 빈 배열이 아닌 레코드 기준")
r = next(x for x in RUNS["a"] if x["clause_id"] == DEMO)
print(f"\n{DEMO} (v127a): 시도 {r['n_attempts']}회 · 최종 관문",
      {g: r["final"][g] for g in GATES})

                   v127a     v127b
⑦ 빈 배열             25.7%     25.7%
json_ok            95.2%     98.1%
schema_ok          90.5%     88.6%
catalog_ok         84.6%     84.6%
quote_ok           85.9%     89.7%
exec_ok            83.3%     84.6%

③④⑤ 는 빈 배열이 아닌 레코드 기준

KBNR-S2-R05 (v127a): 시도 3회 · 최종 관문 {'json_ok': True, 'schema_ok': True, 'catalog_ok': False, 'quote_ok': True, 'exec_ok': False}


### 5.2 뜻 — ⑥ 실행 결과 일치

관문을 다 통과해도 뜻이 틀릴 수 있다. 수수료 면제를 «한도 0원»으로 우회한 출력이 ①~⑤ 를 전부
통과한 적이 있다. 그래서 정답이 있어야 하는 축을 따로 둔다.

**⑥ = 정답 rule 과 모델 rule 을 같은 가상 고객 프로필들에 실행해 결과(금리 · 면제 · 자격)가 같은 비율.**
금리 계산(`resolver.py`)은 여기서 **채점 도구**로만 쓴다 — 표기가 달라도 뜻이 같으면 맞다고 보기 위해서다.

비교 기준선은 **빈 출력** — 모델이 아무것도 안 냈을 때의 ⑥ 이다. 빈 배열이 정답인 조문이 많아서
기준선이 높다.

In [9]:
import kb10six

SIX = {k: kb10six.six(f"showcase/results_kb10v127{k}.json") for k in "ab"}
P = kb10six.P
empty = {c: kb10six.score(g.get("rules") or [], [], "SUM", P, kb10six.BASE_RATE[c.split("-")[0]])["rate"] * 100
         for c, g in kb10six.DRAFT.items()}
avg = lambda d: sum(d.values()) / len(d)

print(f"⑥ (kb10 {len(empty)}조문 평균)   v127a {avg(SIX['a']):.1f} · v127b {avg(SIX['b']):.1f} · 빈 출력 {avg(empty):.1f}\n")
for k in "ab":
    win = sum(SIX[k][c] > empty[c] for c in empty)
    lose = sum(SIX[k][c] < empty[c] for c in empty)
    print(f"v127{k}: 빈 출력보다 나은 조문 {win} · 못한 조문 {lose} · 같음 {len(empty) - win - lose}")
print(f"\n{DEMO}: v127a {SIX['a'][DEMO]:.0f} · v127b {SIX['b'][DEMO]:.0f} · 빈 출력 {empty[DEMO]:.0f}")

⑥ (kb10 105조문 평균)   v127a 82.4 · v127b 83.3 · 빈 출력 87.4

v127a: 빈 출력보다 나은 조문 4 · 못한 조문 11 · 같음 90
v127b: 빈 출력보다 나은 조문 5 · 못한 조문 11 · 같음 89

KBNR-S2-R05: v127a 0 · v127b 0 · 빈 출력 50


예시 조문에서 모델이 무엇을 냈는지 본다. 두 판 모두 ⑥ 0 — 아무것도 안 낸 것(50)보다 나쁘다.

In [10]:
for k in "ab":
    r = next(x for x in RUNS[k] if x["clause_id"] == DEMO)
    print(f"── v127{k}: rule {len(r['output']['rules'])}개")
    for rule in r["output"]["rules"]:
        print(f"   {rule['label']:<14} effect {rule['effect']}")
    print("   관문 오류:", *sorted(set(e.split('. 쓸 수 있는')[0] for e in r["errors"])), sep="\n     ")

── v127a: rule 2개
   우대이율 적용 조건     effect {'type': 'RATE_ADD', 'value': 1.9, 'unit': '%p'}
   우대이율 적용 한도     effect {'type': 'RATE_OVERRIDE', 'value': 1.9, 'unit': '%p'}
   관문 오류:
     SALARY_TRANSFER 에 없는 속성: in
     실행 불가: DslError: 미바인딩 참조: $window. let 에 먼저 묶어야 한다: {"let": {"window": {...}}}
     카탈로그에 없는 변수: SALARY_TRANSFER.min_amount
── v127b: rule 2개
   우대이율 적용 조건     effect {'type': 'RATE_ADD', 'value': 1.9, 'unit': '%p'}
   우대이율 적용 한도     effect {'type': 'RATE_OVERRIDE', 'value': 0}
   관문 오류:
     RATE_OVERRIDE 에 필요한 키가 없다: unit
     SALARY_TRANSFER 에 없는 속성: in
     실행 불가: DslError: 미바인딩 참조: $window. let 에 먼저 묶어야 한다: {"let": {"window": {...}}}
     카탈로그에 없는 변수: SALARY_TRANSFER.source


정답과 견주면 실패가 셋이다.

* **기간** — 정답은 «직전 3개월 중 2개월 이상»인데 모델은 기간을 직전 1개월로 잡았다(한 달 안에서 2개월을 셀 수 없다).
* **표기** — 급여이체 종류 필터를 카탈로그 밖의 변수로 썼고, 둘째 rule 은 `let` 없이 `$window` 를 참조했다
  (관문 ③⑤ 에서 걸림, 재시도 3회로도 못 고침).
* **지어냄** — «평균잔액 100만원 이하분에만 우대»는 표현할 effect 가 없어 정답이 `UNSUPPORTED` 로 둔 문장인데,
  모델은 잔액 조건 rule 을 만들었다. 이 «옮길 수 없는 것을 옮기는» 실패가 kb10 손실의 주된 꼴이다.

**읽는 법.** 형식은 대부분 풀렸고 뜻은 아직이다. 평균 ⑥ 은 빈 출력보다 낮다 — 빈 배열이 정답인
조문에서 rule 을 지어내는 손실이, 규칙이 있어야 하는 조문에서 맞힌 이득보다 크다.
남은 손실의 큰 몫은 «옮길 수 없는 금리 표 · 서술에 금리 rule 을 지어내는» 것이다.

판정은 평균 하나로 하지 않는다. 같은 코드도 판마다 흔들리므로(위 두 판의 차이가 그것이다)
**처방이 닿는 조문의 합**과 두 판 이상으로 본다 — 규칙은 저장소 `METHOD.md`.

---
## 6. 3번 모델로 넘기는 것

### 6.1 결정 (2026-10-08)

* 서비스는 **기본 정보만 받는다** — 가입 상품 등록(가입일 · 만기일 · 납입액 · 명의)과 판정에 필요한 몇 가지.
  **거래 내역은 모으지 않는다.**
* 거래 조건(급여이체 · 카드 사용 · 자동이체 …)은 «충족한다고 가정»하지 않는다. 가정하면 혜택이 부풀고,
  불리한 조건(무거래 → 거래중지)은 뜻이 뒤집힌다. **«조건부»로 넘긴다.**
* 문장은 만들지 않는다. 조건식 · 효과 · 원문 인용 · 모자란 입력을 JSON 으로 넘기고, 3번 모델이 문장으로 옮긴다.

### 6.2 `explain.py` — 고객 입력으로 걸러 낸 조건부 묶음

rule 마다 실행기(`engine.py`)로 판정한다. 입력하지 않은 값은 «없다»가 아니라 **«모른다(UNKNOWN)»** 다.

| `status` | 뜻 |
|---|---|
| `APPLIES` | 입력만으로 충족이 확정 |
| `NOT_APPLICABLE` | 입력만으로 불충족이 확정 |
| `CONDITIONAL` | 모르는 값이 남음 — 아는 갈래를 걷어 낸 조건식 + `missing_inputs` · `missing_events` |

`side` 는 고객에게 유리한지(`BENEFIT`) 불리한지(`CAUTION`)다.

In [11]:
import explain

base = kb10six.BASE_RATE["KBNR"]
ctx = explain.user_ctx("2026-10-12", start="2025-03-02", maturity="2028-03-02", monthly=200_000, base_rate=base)
pack = explain.explain(explain.load_gold("KBNR"), ctx, base)

print(f"{PRODUCTS['KBNR']['product']} · 입력: 가입일 · 만기일 · 월 납입액\n")
print(f"{'rule':<17}{'status':<16}{'side':<9}{'effect':<14}모자란 입력")
print("-" * 86)
for it in pack["rules"]:
    miss = ", ".join(it.get("missing_inputs", []) + [f"events.{e}" for e in it.get("missing_events", [])])
    print(f"{it['rule_id']:<17}{it['status']:<16}{it['side']:<9}{(it['effect'] or {}).get('type', ''):<14}{miss}")

KB나라사랑우대통장 · 입력: 가입일 · 만기일 · 월 납입액

rule             status          side     effect        모자란 입력
--------------------------------------------------------------------------------------
KBNR-S2-R01-01   CONDITIONAL     CAUTION  DISQUALIFY    customer.has_product
KBNR-S2-R01-02   CONDITIONAL     CAUTION  DISQUALIFY    customer.holder_type
KBNR-S2-R05-01   CONDITIONAL     BENEFIT  RATE_ADD      events.SALARY_TRANSFER
KBNR-S2-R09-01   CONDITIONAL     BENEFIT  FEE_WAIVE     events.AUTO_TRANSFER, events.CARD_USAGE
KBNR-S2-R09-02   CONDITIONAL     BENEFIT  FEE_WAIVE     events.AUTO_TRANSFER, events.CARD_USAGE
KBNR-S2-R09-03   NOT_APPLICABLE  BENEFIT  FEE_WAIVE     
KBNR-S2-R09-04   NOT_APPLICABLE  BENEFIT  FEE_WAIVE     


예시 조문의 rule 이 실제로 넘어가는 모양이다. 급여이체는 거래 내역이라 늘 `CONDITIONAL` 로 남고,
3번 모델은 이것을 «급여이체 실적이 있으면 연 1.9%p 우대»처럼 조건부 문장으로 옮긴다.

In [12]:
print(json.dumps(next(it for it in pack["rules"] if it["rule_id"] == f"{DEMO}-01"), ensure_ascii=False, indent=1))

{
 "clause_id": "KBNR-S2-R05",
 "rule_id": "KBNR-S2-R05-01",
 "status": "CONDITIONAL",
 "side": "BENEFIT",
 "condition": {
  ">=": [
   {
    "count_distinct_months": {
     "event": "SALARY_TRANSFER",
     "within": {
      "ref": "$window"
     }
    }
   },
   2
  ]
 },
 "effect": {
  "type": "RATE_ADD",
  "value": 1.9,
  "unit": "%p",
  "applies_to": "BASE_RATE"
 },
 "quote": "▪ 우대이율 : 연 1.9%p ▪ 적용대상 : 결산일의 전월 말일을 기준으로 직전 3개월 동안 2개월 이상 이 통장으로 아래 한 항목 이상에 해당하는 급여이체 실적이 있는 경우",
 "let": {
  "window": {
   "month_range": {
    "from": {
     "anchor": "contract.reference_date",
     "offset_months": -3,
     "day": "first"
    },
    "to": {
     "anchor": "contract.reference_date",
     "offset_months": -1,
     "day": "last"
    }
   }
  }
 },
 "missing_inputs": [],
 "missing_events": [
  "SALARY_TRANSFER"
 ]
}


### 6.3 `fields.py` — 무엇을 물어야 하나

조건식이 읽는 필드를 상품별로 모아, 어디서 얻는 값인지 나눈다. 3번 모델(또는 화면)이 «무엇을 물을지»
정할 때 쓴다.

| 갈래 | |
|---|---|
| `AUTO` | 입력 없이 정해진다 (판정일 · 기본이율) |
| `REGISTRATION` | 상품 등록 화면에서 받는다 |
| `ASK` | 한 번 물으면 된다 |
| `TERMINATION` | 해지할 때 생긴다 — 가입 중에는 늘 모른다 |
| `NOT_COLLECTED` | 거래 내역 · 계좌 상태 — 모으지 않는다 |

In [13]:
import fields

ORDER = (fields.AUTO, fields.REGISTRATION, fields.ASK, fields.TERMINATION, fields.NOT_COLLECTED)

f, _ = fields.product_fields(explain.load_gold("KBNR"))
print(f"{PRODUCTS['KBNR']['product']} — 조건식이 읽는 필드\n")
for name, x in f.items():
    print(f"  {x['source']:<14}{name:<30}{x['desc']}")

tot = collections.Counter()
for pre in explain.BASE_RATE:
    tot.update(fields.product_fields(explain.load_gold(pre))[1])
print(f"\nkb10 정답의 OK rule {sum(tot.values())}개를 «판정에 필요한 가장 먼 갈래»로 셈")
for k in ORDER:
    print(f"  {k:<14}{tot[k]:>3}")
print("\nNOT_COLLECTED · TERMINATION 에 걸린 rule 은 가입 중에는 늘 조건부로 넘어간다")

KB나라사랑우대통장 — 조건식이 읽는 필드

  AUTO          contract.reference_date       판별 기준일
  REGISTRATION  contract.start_date           가입일
  REGISTRATION  customer.holder_type          명의 유형
  ASK           customer.has_product          해당 상품 보유 여부
  NOT_COLLECTED events.AUTO_TRANSFER          자동이체 · 공과금이체
  NOT_COLLECTED events.CARD_USAGE             카드 결제실적(현금서비스 포함)
  NOT_COLLECTED events.SALARY_TRANSFER        급여이체

kb10 정답의 OK rule 65개를 «판정에 필요한 가장 먼 갈래»로 셈
  AUTO            8
  REGISTRATION   24
  ASK            11
  TERMINATION     7
  NOT_COLLECTED  15

NOT_COLLECTED · TERMINATION 에 걸린 rule 은 가입 중에는 늘 조건부로 넘어간다


---
## 7. 지금까지 배운 것 (이력 요약)

8월 말부터 10-06 까지 처방 라운드를 v1 ~ v127 돌렸다. 숫자와 판정은 `EXPERIMENTS.md`,
현황은 `STATUS.md`, 판정 규칙과 그 규칙을 배운 실패는 `METHOD.md` 에 있다.
초기(8/27 ~ 9/13) 서술판 노트북은 `history/qwen3-8B_20260913.ipynb` 로 옮겨 두었다.

* **형식은 프롬프트로 풀렸다.** 연산자 이름을 키로 쓰는 표기를 «올바름/틀림» 쌍으로 보여 주기,
  들여쓴 few-shot, 재시도 중 가장 나은 시도 채택 — 관문이 크게 올랐다. 반대로 모델이 원래 안 쓰던 표현을
  금지어로 부르면 오히려 늘었다.
* **뜻은 프롬프트로 잘 안 풀린다.** 막으면 다른 도구로 샌다 — 표현할 자리(카탈로그)가 없는 것이 원인이었다.
  설명 줄은 어휘를 옮기지만 뜻은 못 옮긴다(`>= 0` 같은 헛조건).
* **두 통장에 맞춘 처방은 다른 상품에 안 들었다.** 그래서 kb10 정답을 만들어 상품 무관 처방만 남겼다.
* **여러 번 측정이 틀렸다.** 참조표 중복 채점, 무출력이 빈 배열 만점을 받던 것, 판정선을 한 번 잰 값으로
  쓴 것 — 모델보다 채점기를 먼저 의심한다.
* **예시 출력에 키 하나만 더해도 빈 배열 비율이 흔들린다** (v125 되돌림).

---
## 8. 한계와 팀이 정할 것

* **넘길 형식** — `explain.py`(판정 JSON) 와 `export_manual.py`(팀 서식 `manual_rule_ko`) 중 무엇을 3번 모델에
  넘길지 정해지지 않았다.
* **기준 문서** — 특약과 상품설명서가 다를 때 어느 쪽을 따를지(예: KB스타통장 수수료 면제).
* **표현 못 하는 조건** — 잔액 구간별 적용, 금리 표 일부 등은 `UNSUPPORTED` 로 남는다.
* **거래 내역 조건**은 설계상 늘 조건부다. 판정이 끝나는 건 기본 정보로 되는 rule 뿐이다.
* 모델은 바꾸지 않고 파인튜닝도 하지 않는다(추론만, QLoRA 는 팀원 몫).